<a href="https://colab.research.google.com/github/emisseldine/MisselBooks/blob/main/LinearAlgebra/workbooks/lqr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center>
<p style="text-align:center">
    <a href="https://raw.githubusercontent.com/emisseldine/OpenLinear/main/BackMatter/logo.png" target="_blank">
    <img src="https://raw.githubusercontent.com/emisseldine/OpenLinear/main/BackMatter/logo.png" width="200" alt="Missel Books Logo">
    </a>
</p>


# 4.10 Control of Linear Dynamical Systems
</center>

---


*To execute your code, hit the play/execute/run button or Shift+Enter.*


1. Consider the dynamical system:

$A = \begin{bmatrix}0.99&0.03&-0.02&-0.31\\0.01&0.47&4.70&0.00\\0.02&-0.06&0.40&0.00\\0.01&-0.04&0.72&0.99\end{bmatrix},$
$B = \begin{bmatrix}0.01&0.99\\-3.44&1.66\\-0.83&0.44\\-0.47&0.25\end{bmatrix},$
$C = \begin{bmatrix}1&0&0&0\\0&1&0&0\\0&0&1&0\\0&0&0&1\end{bmatrix},$
$\boldsymbol{a} = \begin{bmatrix}0\\0\\0\\1\end{bmatrix},$
$\boldsymbol{b} = \begin{bmatrix}0\\0\\0\\0\end{bmatrix},$
$T=120$.

In [ ]:
import numpy as np

def lstsq_constraint(A, b, C, d):   #|Ax-b|^2 is the objective to minimize
    ATA = A.T @ A                   #subject to the constraint Cx=d
    CT = (C.T).reshape((len(ATA), -1))
    p = len(CT[0])
    KKT = np.block([[2*ATA, CT], [CT.T, np.zeros((p, p))]])
    KKTb = np.hstack((2*A.T @ b, d))
    return np.linalg.solve(KKT, KKTb)[:-p]

#Linear Quadratic Regulator
def lqr(A, B, C, a, b, T, rho):
    n = len(A)
    Bmtx,Cmtx = B.reshape((n,-1)),C.reshape((-1,n))
    m,p = len(Bmtx[0]),len(Cmtx)
    Ablock = np.block([
        [np.kron(np.eye(T+1),Cmtx),np.zeros((p*(T+1),m*T))],
        [np.zeros((m*T,n*(T+1))),np.sqrt(rho)*np.eye(m*T)]])
    Cblock = np.block([
            [np.kron(np.eye(T,T+1),A) - np.kron(np.eye(T,T+1,1),np.eye(n)),
                  np.kron(np.eye(T),Bmtx)],
            [np.kron(np.vstack([np.eye(1,T+1,0),np.eye(1,T+1,T)]),np.eye(n)),
                  np.zeros((2*n,m*T))]])
    dblock = np.concatenate((np.zeros(n*T),a,b))
    z = lstsq_constraint(Ablock,np.zeros(len(Ablock)),Cblock,dblock)

    x = np.array([z[i*n:(i+1)*n] for i in range(T+1)])
    u = np.array([z[n*(T+1)+(i)*m: n*(T+1)+(i+1)*m] for i in range(T)])
    y = np.array([C @ xt for xt in x])
    return x, u, y

## Parameters #######
# x_(t+1) = Ax_t + Bu_t
A = # enter the transition matrix A
B = # enter the input matrix B
# y_t = Cx_t
C = # enter the output matrix C
a = # x_0
b = # x_T
T = # the number of iterations in the time series
print(A,'\n',B,'\n',C,'\n',a,'\n',b,'\n',T)

2.   Simulate the dynamical system without control.

In [ ]:
## Simulations #######
import matplotlib.pyplot as plt

## No Control ##
Xno = np.zeros((T+1,len(a)))  #initialize Xno
Xno[0] = a
for k in range(T):             #simulation without control
    Xno[k+1] = A @ Xno[k]
# plot output vector y_t (there is no control vector u_t)
Yno = np.array(C @ Xno.T).T
for yi in range(len(Yno[0])):
    plt.plot(np.arange(T+1),Yno[:,yi],label=f"y_t{yi}")
plt.ylabel("y_t")
plt.xlabel("t\n No Control")
plt.ylim(-4,3)
plt.legend()
plt.show()

3.   Simulate the dynamical system with linear quadratic control.

In [ ]:
## Linear Quadratic Control ##
rho = 100
#initialize sequences X (states), U (input/control), Y (output)
X,U,Y = lqr(A,B,C,a,b,T,rho)
# plot input vectors u_t
plt.plot(np.arange(T),U,label=[f"u_t{i}" for i in range(len(U[0]))])
plt.xlabel('t\n rho = ' + str(rho))
plt.ylabel('u_t')
plt.ylim(-0.05,0.25)
plt.legend()
plt.show()
# plot output vectors y_t
plt.plot(np.arange(T+1),Y,label=[f"y_t{i}" for i in range(len(Y[0]))])
plt.xlabel('t\n rho = ' + str(rho))
plt.ylabel('y_t')
plt.ylim(-4,3)
plt.legend()
plt.show()

<details><summary>Click here for the solution</summary>

Predicted 2020 Humman Population: 113.6 million

Predicted 2020 Zombie Population: 13.9 million

<details><summary>Click here for the Transition Matrix only if you get stumped.</summary>

```Python
A = np.vstack((np.block([b,0]),
              np.block([inf*np.diag(1-d[:99]),np.zeros((99,3))]),
              inf*(1-d[99])*np.eye(1,102,99) + inf*(1-d[100])*np.eye(1,102,100),
               np.block([d,0.95])))
```

</details>

</details>

4. Simulate the dynamical system with linear state feedback control.

In [ ]:
#Linear State Feedback Control
KT = np.zeros(B.shape)             #initialize/compute state-feedback gain matrix
for  i in range(len(A)):
    X,U,Y = lqr(A,B,C,np.eye(1,len(A),i)[0],b,T,rho)
    KT[i] = U[0]
K = KT.T

ABK = A + B@K    #form matrix A + BK, outer product used to avoid having to reshape vectors as matrices
Xst = np.zeros((T+1,len(A)))   #initialize Xst
Xst[0] = a
for k in range(T):         #simulation with state-feedback control
    Xst[k+1] = ABK @ Xst[k]
Yst = np.array(C @ Xst.T).T
# plot input vectors u_t
plt.plot(np.arange(T), [K @ x for x in Xst[:-1]],label=[f"u_t{i}" for i in range(len(U[0]))])
plt.xlabel("t\n State Feedback Control")
plt.ylabel('u_t')
plt.ylim(-0.05,0.25)
plt.legend()
plt.show()
# plot output vectors y_t
plt.plot(np.arange(T+1),Yst,label=[f"y_t{i}" for i in range(len(Y[0]))])
plt.xlabel("t\n State Feedback Control")
plt.ylabel("y_t")
plt.ylim(-4,3)
plt.legend()
plt.show()

5. How do these three simulations compare?

In [ ]:
## Analysis #######
print() #How do these three simulations compare?

---
*When completed, convert your .ipynb notebook into an .html webpage and upload it to the course's learning management system, e.g. Canvas.*
<details><summary> Instructions to convert to .html </summary>

1. *Download the .ipynb notebook, which you want to convert, on your local computer.*
2. *Run the code below to upload the .ipynb notebook.*
3. *The .html webpage will be downloaded automatically on your local machine.*

</details>

In [ ]:
from google.colab import files
f = files.upload()
import subprocess
file0 = list(f.keys())[0]
_ = subprocess.run(["pip", "install", "nbconvert"])
_ = subprocess.run(["jupyter", "nbconvert", file0, "--to", "html"])
files.download(file0[:-5]+"html")



---
<center>

##### This Python workbook was produced for [Linear Algebra Done Openly](https://github.com/emisseldine/MisselBooks/tree/main/LinearAlgebra) by Dr. Andrew Misseldine (2026).
<p style="text-align:center">
    <a href="https://raw.githubusercontent.com/emisseldine/OpenLinear/main/BackMatter/logo.png" target="_blank">
    <img src="https://raw.githubusercontent.com/emisseldine/OpenLinear/main/BackMatter/logo.png" width="200" alt="Missel Books Logo">
    </a>
</p>
</center>